<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## 03-1 · Part 4: Reduce Squared Errors

**Compare all four clock errors together and see how a changed observation affects the best correction.**

[Part 2](02_single_variable_analytical_optimization.ipynb) found the best constant correction, and [Part 3](03_multivariable_analytical_optimization.ipynb) added a rate setting. We return to one correction and apply the result already proved. We now focus on what the squared-error score counts and how the answer changes with the data.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Why add squared errors?

The recorded errors are $-2,-1,-1,0$ min. Adding a correction $q=1$ min leaves $-1,0,0,1$ min.
Adding those remaining errors gives zero, even though two readings are still wrong.
Squaring prevents positive and negative errors from cancelling:

$$
(-1)^2+0^2+0^2+1^2=2.
$$

Compare the two totals for the same four remaining errors. The signed errors cancel on the left. On the right, the errors of $-1$ and $+1$ min each contribute one square, so neither error disappears from the score.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-1_unconstrained_optimization_analytical_methods/assets/signed_errors_and_squares.png" alt="Remaining errors minus 1, 0, 0, plus 1 minutes have signed sum zero. Squaring gives contributions 1, 0, 0, 1 and a total of 2 square minutes. The two nonzero contributions are shown as equal square tiles." width="800" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

**Least squares** means choosing the settings that make the sum of squared errors as small as possible.
Here the remaining errors are $q-2,q-1,q-1,q$, so the score is

$$
\phi(q)=(q-2)^2+(q-1)^2+(q-1)^2+q^2.
$$

The score has units of min². It measures total squared error, not the signed total or the largest individual error.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Inspect the signed errors and their squares for one correction. Keep both results in the same evaluation record.


In [ ]:
import numpy as np

OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min


def evaluate_clock(correction, observed_errors=OBSERVED_ERRORS):
    remaining_errors = np.asarray(observed_errors) + correction
    return {
        "decision": correction,
        "response": remaining_errors,
        "squared_errors": remaining_errors ** 2,
        "objective": float(np.sum(remaining_errors ** 2)),
    }


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The signed total can be zero even when individual readings are wrong. The squared-error total should be 2 min².


In [ ]:
result = evaluate_clock(1.0)
print("Remaining errors:", result["response"], "min")
print("Signed total:", np.sum(result["response"]), "min")
print("Squared contributions:", result["squared_errors"], "min²")
print("Sum of squares:", result["objective"], "min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Use the result already found

Part 2 showed that the best constant correction cancels the mean recorded error:

$$
q^\star=-\frac{-2-1-1+0}{4}=1\quad\text{min}.
$$

The star marks the best correction. It leaves errors $-1,0,0,1$ min and a score of 2 min². The code checks this result directly from the original errors.

Because the objective is quadratic and there are no constraints, this example is also an **unconstrained quadratic program (QP)**. This name describes the same problem we have already solved.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Use the mean-error formula, then check the score from the original errors.


In [ ]:
best_correction = -float(np.mean(OBSERVED_ERRORS))
best_result = evaluate_clock(best_correction)
print("Best correction:", best_correction, "min")
print("Remaining errors:", best_result["response"], "min")
print("Minimum score:", best_result["objective"], "min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Compare the whole error pattern

**Question.** At $q=2$, the remaining errors are $0,1,1,2$ min. The first reading is now correct. Is this better overall than $q=1$, which leaves $-1,0,0,1$ min?

**Solution.** No. At $q=2$, the score is $0+1+1+4=6$ min², compared with $1+0+0+1=2$ min² at $q=1$. Improving one reading can make the total score worse.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Compare the two corrections in the worked example. Inspect all four squared contributions, not just the first one.


In [ ]:
for correction in (1.0, 2.0):
    result = evaluate_clock(correction)
    print(f"q = {correction:.1f} min")
    print("  Squared contributions:", result["squared_errors"], "min²")
    print("  Total:", result["objective"], "min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Change one recorded error

Suppose only the last observation changes from 0 to 3 min, giving $-2,-1,-1,3$ min.
The correction rule and squared-error objective stay the same. Predict the best correction before calculating.

**Worked answer.** The new mean is $(-2-1-1+3)/4=-0.25$ min, so add 0.25 min.
The remaining errors are $-1.75,-0.75,-0.75,3.25$ min and their squares sum to 14.75 min².
The larger last observation changes the best correction.


In [ ]:
changed_errors = OBSERVED_ERRORS.copy()
changed_errors[-1] = 3.0
changed_correction = -float(np.mean(changed_errors))
changed_result = evaluate_clock(changed_correction, changed_errors)
print("New observations:", changed_errors, "min")
print("Best correction:", changed_correction, "min")
print("Remaining errors:", changed_result["response"], "min")
print("Minimum squared-error score:", changed_result["objective"], "min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

[Part 5](05_convexity_and_global_minima.ipynb) explains how convexity supports global verification of an analytical solution. Lecture 03-2 then keeps the two-setting squared-error problem and adds one exact calibration requirement.
The objective remains quadratic. Adding a constraint changes which settings are acceptable.
